# 05-02 트랜스포머로 띄어쓰기 학습하기

런타임 유형을 GPU로 바꾼 뒤 첫 셀부터 차례로 실행하세요. 전체 실행에 T4 GPU로 몇 분 걸려요.

In [ ]:
import json
import math
import os
import random
import re
import time
import urllib.request

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)
print("device:", device)

## 0. 준비: 05-01의 데이터 코드

05-01에서 만든 코드를 한 셀에 모았어요.

In [ ]:
URL = "https://raw.githubusercontent.com/KLUE-benchmark/KLUE/main/klue_benchmark/klue-nli-v1.1/klue-nli-v1.1_train.json"
if not os.path.exists("klue_nli_train.json"):
    urllib.request.urlretrieve(URL, "klue_nli_train.json")
with open("klue_nli_train.json", encoding="utf-8") as f:
    data = json.load(f)
raw = sorted({s for d in data for s in (d["premise"], d["hypothesis"])})

sentences = [re.sub(r"\s+", " ", s).strip() for s in raw]
sentences = [s for s in sentences if 10 <= len(s) <= 120 and " " in s]
random.seed(0)
random.shuffle(sentences)
n = len(sentences)
train_s, val_s, test_s = sentences[: int(n * 0.9)], sentences[int(n * 0.9) : int(n * 0.95)], sentences[int(n * 0.95) :]

def make_example(sentence):
    chars, labels = [], []
    for i, ch in enumerate(sentence):
        if ch == " ":
            continue
        chars.append(ch)
        labels.append(1 if i + 1 < len(sentence) and sentence[i + 1] == " " else 0)
    return "".join(chars), labels

def apply_spacing(text, labels):
    return "".join(ch + (" " if y == 1 else "") for ch, y in zip(text, labels)).strip()

class CharTokenizer:
    def __init__(self, texts, special=("<pad>", "<unk>")):
        chars = sorted(set("".join(texts)))
        self.itos = list(special) + chars
        self.stoi = {t: i for i, t in enumerate(self.itos)}
        self.pad_id, self.unk_id = self.stoi["<pad>"], self.stoi["<unk>"]

    def __len__(self):
        return len(self.itos)

    def encode(self, text):
        return [self.stoi.get(ch, self.unk_id) for ch in text]

    def decode(self, ids):
        return "".join(self.itos[i] for i in ids if i != self.pad_id)

class SpacingDataset(Dataset):
    def __init__(self, sentences, tok):
        self.items = [(tok.encode(t), y) for t, y in map(make_example, sentences)]

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        return self.items[i]

def collate(batch, pad_id=0):
    max_len = max(len(ids) for ids, _ in batch)
    ids = torch.full((len(batch), max_len), pad_id)  # (B, T)
    labels = torch.full((len(batch), max_len), -100)  # (B, T)
    for i, (x, y) in enumerate(batch):
        ids[i, : len(x)] = torch.tensor(x)
        labels[i, : len(y)] = torch.tensor(y)
    return ids, labels

tok = CharTokenizer([make_example(s)[0] for s in train_s])
train_dl = DataLoader(SpacingDataset(train_s, tok), batch_size=64, shuffle=True, collate_fn=collate)
val_dl = DataLoader(SpacingDataset(val_s, tok), batch_size=256, collate_fn=collate)
test_dl = DataLoader(SpacingDataset(test_s, tok), batch_size=256, collate_fn=collate)
print(len(train_s), len(val_s), len(test_s), "어휘집", len(tok))

## 1. 모델: 04-04의 인코더 + 분류 레이어

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.n_heads, self.d_head = n_heads, d_model // n_heads
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

    def split(self, t):
        B, T, _ = t.shape
        return t.view(B, T, self.n_heads, self.d_head).transpose(1, 2)  # (B, H, T, d_head)

    def forward(self, x, mask=None):
        B, T, d_model = x.shape
        q, k, v = self.split(self.q_proj(x)), self.split(self.k_proj(x)), self.split(self.v_proj(x))
        scores = q @ k.transpose(-2, -1) / math.sqrt(self.d_head)  # (B, H, T, T)
        if mask is not None:
            scores = scores.masked_fill(~mask, float("-inf"))
        self.weights = F.softmax(scores, dim=-1)
        out = (self.weights @ v).transpose(1, 2).reshape(B, T, d_model)  # (B, T, d_model)
        return self.out_proj(out)

class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.norm1, self.norm2 = nn.LayerNorm(d_model), nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, n_heads)
        self.ff = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model), nn.Dropout(dropout))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        x = x + self.dropout(self.attn(self.norm1(x), mask))
        return x + self.ff(self.norm2(x))  # (B, T, d_model)

class Encoder(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, n_layers, d_ff, max_len, dropout=0.1):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)
        self.blocks = nn.ModuleList([TransformerBlock(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.norm = nn.LayerNorm(d_model)

    def forward(self, ids, mask=None):
        T = ids.shape[1]
        x = self.tok_emb(ids) + self.pos_emb(torch.arange(T, device=ids.device))  # (B, T, d_model)
        for block in self.blocks:
            x = block(x, mask)
        return self.norm(x)

In [ ]:
class SpacingModel(nn.Module):
    def __init__(self, vocab_size, pad_id, d_model=128, n_heads=4, n_layers=3, d_ff=512, max_len=128):
        super().__init__()
        self.pad_id = pad_id
        self.encoder = Encoder(vocab_size, d_model, n_heads, n_layers, d_ff, max_len)
        self.head = nn.Linear(d_model, 2)  # 0: 붙여 쓰기, 1: 뒤에 띄어 쓰기

    def forward(self, ids):
        # ids: (B, T)
        mask = (ids != self.pad_id)[:, None, None, :]  # (B, 1, 1, T), 패딩은 보지 않아요
        h = self.encoder(ids, mask)  # (B, T, d_model)
        return self.head(h)  # (B, T, 2)

torch.manual_seed(0)
model = SpacingModel(len(tok), tok.pad_id).to(device)
print("파라미터 수:", sum(p.numel() for p in model.parameters()))
ids, labels = next(iter(train_dl))
print(ids.shape, "->", model(ids.to(device)).shape)

## 2. 평가 함수

In [ ]:
@torch.no_grad()
def evaluate(model, dl):
    model.eval()
    tp = fp = fn = correct = total = 0
    loss_sum = 0.0
    for ids, labels in dl:
        ids, labels = ids.to(device), labels.to(device)
        logits = model(ids)  # (B, T, 2)
        loss_sum += F.cross_entropy(logits.reshape(-1, 2), labels.reshape(-1), ignore_index=-100, reduction="sum").item()
        pred = logits.argmax(-1)  # (B, T)
        valid = labels != -100
        correct += ((pred == labels) & valid).sum().item()
        total += valid.sum().item()
        tp += ((pred == 1) & (labels == 1)).sum().item()
        fp += ((pred == 1) & (labels == 0)).sum().item()
        fn += ((pred == 0) & (labels == 1)).sum().item()
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-9)
    return {"loss": loss_sum / total, "acc": correct / total, "precision": precision, "recall": recall, "f1": f1}

print("학습 전:", evaluate(model, val_dl))

## 3. 학습

In [ ]:
torch.manual_seed(0)
model = SpacingModel(len(tok), tok.pad_id).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
epochs = 5
train_losses, val_history = [], []
start = time.time()
for epoch in range(epochs):
    model.train()
    for ids, labels in train_dl:
        ids, labels = ids.to(device), labels.to(device)
        logits = model(ids)  # (B, T, 2)
        loss = F.cross_entropy(logits.reshape(-1, 2), labels.reshape(-1), ignore_index=-100)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())
    val = evaluate(model, val_dl)
    val_history.append(val)
    print(f"epoch {epoch + 1}: val loss={val['loss']:.4f}, acc={val['acc']:.4f}, f1={val['f1']:.4f} ({time.time() - start:.0f}s)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].plot(train_losses, color="tab:red", alpha=0.4)
axes[0].set_title("train loss (per step)")
axes[0].set_xlabel("step")
axes[1].plot(range(1, epochs + 1), [v["f1"] for v in val_history], "o-", color="tab:orange", label="F1")
axes[1].plot(range(1, epochs + 1), [v["acc"] for v in val_history], "o-", color="tab:blue", label="accuracy")
axes[1].set_title("validation")
axes[1].set_xlabel("epoch")
axes[1].legend()
plt.show()

## 4. 테스트 데이터로 평가하기

In [ ]:
test = evaluate(model, test_dl)
print({k: round(v, 4) for k, v in test.items()})

# 비교 기준: 한 번도 띄우지 않는 모델
labels_all = [y for s in test_s for y in make_example(s)[1]]
print(f"항상 0으로 답하면 정확도 {1 - sum(labels_all) / len(labels_all):.4f}, F1 0")

## 5. 띄어쓰기 복원하기

In [ ]:
@torch.no_grad()
def restore_spacing(text):
    model.eval()
    text = text.replace(" ", "")
    ids = torch.tensor([tok.encode(text)], device=device)  # (1, T)
    probs = F.softmax(model(ids), dim=-1)[0, :, 1]  # (T,), 뒤에 띄어 쓸 확률
    return apply_spacing(text, (probs > 0.5).long().tolist()), probs.cpu()

for s in test_s[:5]:
    pred, _ = restore_spacing(s)
    print("정답:", s)
    print("예측:", pred)
    print("일치" if pred == s else "다름")
    print()

In [ ]:
exact = sum(restore_spacing(s)[0] == s for s in test_s) / len(test_s)
print(f"문장 전체가 정답과 같은 비율: {exact:.3f}")

In [ ]:
for s in ["아버지가방에들어가신다", "나는오늘학교에서친구를만났다", "파이토치로띄어쓰기모델을만들었다"]:
    print(restore_spacing(s)[0])

In [ ]:
text = "나는오늘학교에서친구를만났다"
pred, probs = restore_spacing(text)
plt.figure(figsize=(9, 3))
plt.bar(range(len(text)), probs, color=["tab:orange" if p > 0.5 else "tab:gray" for p in probs])
plt.axhline(0.5, color="k", ls="--", lw=0.8)
plt.xticks(range(len(text)))
plt.xlabel("character index")
plt.ylabel("P(space after)")
plt.title("P(space after each character)")
plt.show()
print(list(enumerate(text)))